In [ ]:
import os
import sys
from pathlib import Path

root_dir = str(Path(os.getcwd()).parent)

if root_dir not in sys.path:
    sys.path.append(root_dir)

output_dir = "Outputs"
os.makedirs(output_dir, exist_ok=True)

<div style="background-color: #217de0; color: white; padding: 30px; border-radius: 0px;">
<h1 style="margin: 0; color: #29007b; font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;">Incompressible Navier-Stokes Solvers </h1>
<p style="margin: 5px 0 0 0; color: #d0e6ff; font-size: 1.2em;">Real-Time High-Fidelity Eulerian Fluid Simulation via Operator Splitting</p>
</div>

Imports:

In [2]:
import numpy as np
from IPython.display import Video, HTML

from Solver.Stable_fluids import *

---

## Incompressible Fluid Dynamics

This framework tracks the evolution of an incompressible velocity field $\mathbf{u} = (u, v)$ and a transported density field (like dye or smoke) governed by the transient **Navier-Stokes Equations**:

$$\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} = -\frac{1}{\rho}\nabla p + \nu \nabla^2 \mathbf{u} + \mathbf{f}$$
$$\nabla \cdot \mathbf{u} = 0$$

To bypass strict CFL stability constraints that typically crash standard explicit solvers, this implementation utilizes **Jos Stam's Stable Fluids** fractional-step operator splitting pipeline:

1. **Add Forces:** Applies external velocity and density inputs (e.g., a continuous dye source).
2. **Advection:** Solves the non-linear transport using an unconditionally stable **Semi-Lagrangian Back-Tracing Scheme**.
3. **Diffusion:** Handles viscous friction and dye diffusion using a stable implicit linear system.
4. **Projection:** Enforces the divergence-free mass conservation constraint ($\nabla \cdot \mathbf{u} = 0$) by computing a scalar Poisson pressure field and subtracting its gradient from the velocity.

---

### Defining the Domain & Conditions:

Domain

In [3]:
H  = 7
N  = 230
dt = 0.1
T  = 10
KINEMATIC_VISCOSITY = 0.001

# Grid
x,y = np.linspace(0, H, N), np.linspace(0, H, N)
X,Y = np.meshgrid(x, y,
                indexing='ij')
COORDINATES = np.stack((X,Y), axis=-1)
dx = H / (N - 1)

# Object defined as Mask
obstacle_mask = make_circular_obstacle(X, Y, center=(H/2, H-H/3), radius=0.75)
w0 = np.zeros(X.shape + (2,))           # initial state (Nx, Ny, 2)

Conditions

In [4]:
nsteps = int(T/dt)
solution = np.zeros((nsteps, N, N, 2))

w = w0.copy()
for i in tqdm(range(nsteps), desc="Simulating"):
    t = i * dt
    w1 = w + dt * force(t, X, Y, 9)
    w1 = apply_obstacle(w1, obstacle_mask)
    w2 = advect(w1, dt)
    w2 = apply_obstacle(w2, obstacle_mask)
    w3 = diffuse(w2, dt, dx)
    w3 = apply_obstacle(w3, obstacle_mask)
    w4 = project(w3)
    w4 = apply_obstacle(w4, obstacle_mask)
    w = w4.copy()
    solution[i] = w

Simulating:   0%|          | 0/100 [00:00<?, ?it/s]


NameError: name 'H' is not defined

### Running the Simulation & Rendering Pipeline:

In [ ]:
target_video = os.path.join(output_dir, "Fluid_Flow_Simulation_2D.mp4")

render_fluid_frames(solution, H, N, dx, dpi=200, num_workers=15, framerate=30,
                    video_filename=target_video)